# Feature Engineering and Feature Selection

## Setup

Run the following code to import the necessary libraries/modules.

In [1]:
## Setup
import pandas as pd
import numpy as np
import os
import math
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import itertools
import sys
import pathlib
import notebook_file_utilities.auto_add_project_root as proroot

# For reading in modules from src and reading/saving files
project_root = proroot.auto_add_project_root()

from src.pipeline.preprocessing import preprocessing
from src.cleaning.merged_tornado_indicator import create_tornado_indicator
import src.cleaning.na_imputer as na_imp
import src.cleaning.drop_features as dfeat
import Data.metadata.cleaned_feature_info as featinfo

The code below reads in processed data, imputes nan values, drops quality code features, and drops duplicates as in the eda notebook.

In [2]:
data = create_tornado_indicator(time_window=1,val_radius=50)

/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:93: DtypeWarning: Columns (7,14,15,16,17,19,20,21,22,23,24,25,26,27,28,29,30,32,33,34,40,41,42,43,44,45,46,47,50,51,52,56,57,58,59,60,65,68,69,70,71,76,79,80,81,82,83,90,91,92,93,94,95,96,97,98,99,102,104,105,106,110,111,113,120,121,122,125) have mixed types. Specify dtype option on import or set low_memory=False.
  station_dfs=[pd.read_csv(os.path.join(station_dir,file)).copy() for file in station_csv_files]
/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:93: DtypeWarning: Columns (39,40,41,42,43,47,48,52,53,54,55,57,58,59,60,61,65,70,71,76,77,88,89,106,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  station_dfs=[pd.read_csv(os.path.join(station_dir,file)).copy() for file in station_csv_files]
/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:93: Dt

In [3]:
data = preprocessing.fit_transform(data)

In [ ]:

data.info() 
# matches eda data info after the na imputation,
# a few feature drops (quality codes), and duplicate deletion.
# Only difference is that this is now streamlined with the 
# preprocessing pipeline AND some particular features
# have been rescaled as per ISD Documentation see mark down 
# below

<class 'pandas.core.frame.DataFrame'>
Index: 551380 entries, 0 to 1626351
Data columns (total 24 columns):
 #   Column                                                         Non-Null Count   Dtype         
---  ------                                                         --------------   -----         
 0   DEW- Air Temperature Observation- Dew Point Temperature        551380 non-null  float64       
 1   TMP- Air Temperature Observation- Air Temperature              551380 non-null  float64       
 2   MA1- Atmospheric Pressure Observation- Altimeter Setting Rate  551380 non-null  float64       
 3   MA1- Atmospheric Pressure Observation- Station Pressure Rate   551380 non-null  float64       
 4   SLP- Atmospheric Pressure Observation- Sea Level Pressure      551380 non-null  float64       
 5   WND- Wind Observation- Speed Rate                              551380 non-null  float64       
 6   CIG- Sky Condition Observation- Ceiling Height Dimension       551380 non-null  float64 

## Necessary Feature Scaling 

There are a few features that are scaled by 10 (or some other value) from the station data we collected. These are summarized in [Data/metadata/cleaned_feature_info.py](<../Data/metadata/cleaned_feature_info.py>). For convenience, we copy all (potentially) scaled features here:

* CIG- Sky Condition Observation- Ceiling Height Dimension : scaled by 1
* DEW- Air Temperature Observation- Dew Point Temperature : scaled by 10
* TMP- Air Temperature Observation- Air Temperature : scaled by 10
* MA1- Atmospheric Pressure Observation- Altimeter Setting Rate : scaled by 10
* MA1- Atmospheric Pressure Observation- Station Pressure Rate : scaled by 10
* SLP- Atmospheric Pressure Observation- Sea Level Pressure : scaled by 10
* VIS- Visibility Observation- Distance Dimension : scaled by 1
* WND- Wind Observation- Direction Angle : scaled by 1
* WND- Wind Observation- Speed Rate : scaled by 10


The following code applies the necessary scaling to convert these values to their proper scale and displays the original column side-by-side for quick verification. Moreover we replace the original column with the appropriately its scaled version

In [ ]:
for col, vals in featinfo.isd_float_meteor_columns.items():
    scale = vals['scale']
    
    rescaled = data[col]*(1/scale)
    display(pd.concat([data[col],rescaled],axis =1))
    data[col]= rescaled
    


,CIG- Sky Condition Observation- Ceiling Height Dimension,CIG- Sky Condition Observation- Ceiling Height Dimension
0,22000.0,22000.0
9,22000.0,22000.0
16,22000.0,22000.0
23,22000.0,22000.0
104,22000.0,22000.0
...,...,...
1626347,22000.0,22000.0
1626348,22000.0,22000.0
1626349,2896.0,2896.0
1626350,3048.0,3048.0


,DEW- Air Temperature Observation- Dew Point Temperature,DEW- Air Temperature Observation- Dew Point Temperature
0,22.0,2.2
9,6.0,0.6
16,22.0,2.2
23,22.0,2.2
104,50.0,5.0
...,...,...
1626347,39.0,3.9
1626348,50.0,5.0
1626349,67.0,6.7
1626350,72.0,7.2


,TMP- Air Temperature Observation- Air Temperature,TMP- Air Temperature Observation- Air Temperature
0,106.0,10.6
9,33.0,3.3
16,44.0,4.4
23,183.0,18.3
104,161.0,16.1
...,...,...
1626347,189.0,18.9
1626348,183.0,18.3
1626349,178.0,17.8
1626350,172.0,17.2


,MA1- Atmospheric Pressure Observation- Altimeter Setting Rate,MA1- Atmospheric Pressure Observation- Altimeter Setting Rate
0,10159.0,1015.9
9,10152.0,1015.2
16,10125.0,1012.5
23,10112.0,1011.2
104,10058.0,1005.8
...,...,...
1626347,10034.0,1003.4
1626348,10034.0,1003.4
1626349,10030.0,1003.0
1626350,10034.0,1003.4


,MA1- Atmospheric Pressure Observation- Station Pressure Rate,MA1- Atmospheric Pressure Observation- Station Pressure Rate
0,9689.0,968.9
9,9682.0,968.2
16,9656.0,965.6
23,9643.0,964.3
104,9591.0,959.1
...,...,...
1626347,9571.0,957.1
1626348,9571.0,957.1
1626349,9567.0,956.7
1626350,9571.0,957.1


,SLP- Atmospheric Pressure Observation- Sea Level Pressure,SLP- Atmospheric Pressure Observation- Sea Level Pressure
0,10160.0,1016.0
9,10153.0,1015.3
16,10126.0,1012.6
23,10109.0,1010.9
104,10054.0,1005.4
...,...,...
1626347,10027.0,1002.7
1626348,10027.0,1002.7
1626349,10024.0,1002.4
1626350,10027.0,1002.7


,VIS- Visibility Observation- Distance Dimension,VIS- Visibility Observation- Distance Dimension
0,16000.0,16000.0
9,16000.0,16000.0
16,16000.0,16000.0
23,16000.0,16000.0
104,16000.0,16000.0
...,...,...
1626347,16093.0,16093.0
1626348,16093.0,16093.0
1626349,16093.0,16093.0
1626350,16093.0,16093.0


,WND- Wind Observation- Direction Angle,WND- Wind Observation- Direction Angle
0,130.0,130.0
9,170.0,170.0
16,150.0,150.0
23,170.0,170.0
104,150.0,150.0
...,...,...
1626347,210.0,210.0
1626348,200.0,200.0
1626349,190.0,190.0
1626350,190.0,190.0


,WND- Wind Observation- Speed Rate,WND- Wind Observation- Speed Rate
0,26.0,2.6
9,26.0,2.6
16,46.0,4.6
23,77.0,7.7
104,62.0,6.2
...,...,...
1626347,77.0,7.7
1626348,93.0,9.3
1626349,57.0,5.7
1626350,46.0,4.6


In [ ]:
temp = data.sort_values(['STATION','STATION_DATE_TIME'])
temp[['STATION_DATE_TIME']]
#[['STATION','STATION_DATE_TIME','MA1- Atmospheric Pressure Observation- Station Pressure Rate']]
pd.concat(temp[['STATION_DATE_TIME']]

,STATION_DATE_TIME
0,2000-01-01 00:00:00
9,2000-01-01 06:00:00
16,2000-01-01 12:00:00
23,2000-01-01 18:00:00
104,2000-01-02 00:00:00
...,...
443050,2005-12-31 19:53:00
443051,2005-12-31 20:53:00
443052,2005-12-31 21:53:00
443053,2005-12-31 22:53:00


## Temporal and Aggregate Features


| Feature                      | Definition                                            | Purpose                                                    |
| ---------------------------- | ----------------------------------------------------- | ---------------------------------------------------------- |
| **Month**, **Hour**          | Extracted from `STATION_DATE_TIME`                    | Capture seasonal and diurnal tornado patterns.             |
| **Lag Variables (optional)** | Previous-hour averages of key continuous features     | Encodes temporal persistence for sequence-aware models.    |
| **Rolling Means (optional)** | e.g., rolling 3-hour averages for `TMP`, `DEW`, `SLP` | Smooth noise, highlight sustained anomalies before events. |

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class (BaseEstimator, TransformerMixin):
    def __init__(self, value=0):
        self.value = value

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.fillna(self.value)


## Feature Engineering Plan

Our next step is Feature Engineering.

| New Feature                     | Definition                                                   | Motivation                                                                        |
| ------------------------------- | ------------------------------------------------------------ | --------------------------------------------------------------------------------- |
| **Temperature–Dewpoint Spread** | `TMP - DEW`                                                  | This is the wet-bulb depression. Together with dewpoint this is often a good indicator for storm occurrence.     |                                     |
| **Pressure Change Sign**   | `                                   | Represents local pressure trend intensity (potential pre-storm signal).           |





The next cell creates Temperature-Dewpoint

## Scaling and Transformation Strategies


| Model Type                                     | Scaling Plan                                                         |
| ---------------------------------------------- | -------------------------------------------------------------------- |
| **Tree-based models (Decision Tree, XGBoost)** | No scaling needed; use raw or log-transformed values where skewed.   |



## Drop Features

| Category                          | Features to Drop                                                                                                                                      | Rationale                            |
| --------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------ |
| **Pressure redundancy**           | `MA1 Altimeter Rate`, `MA1 Station Pressure Rate`                                                                                                     | > 0.99 correlation                   |
| **Thermodynamic redundancy**      | One of `TMP` or `DEW`                                                                                                                                 | +0.84 correlation                    |
| **Geospatial/labeling**           | `STATION_LAT`, `STATION_LON`, `TORNADO_BEGIN_LAT`, `TORNADO_BEGIN_LON`, `TORNADO_END_LAT`, `TORNADO_END_LON`, `TORNADO_INITIAL_DISTANCE_FROM_STATION` | Used to derive label                 |
| **Censored / non-informative**    | `VIS`, `CIG`                                                                                                                                          | Maxed-out values, non-discriminative |
| **Weak / unrepresentative winds** | `WND-Direction Angle`, `WND-Type Code`                                                                                                                | Local-only; low predictive power     |
| **Identifiers**                   | `STATION`                                                                                                                                             | Join key only                        |
